# Guesser evaluation environment check

Use this notebook first to verify the Python 3.12 evaluation environment and its image/scientific packages. For the model/GPU one-batch smoke check, use the [Guesser training README](../../training/README.md#smoke-check); for the A/B training and validation comparison, continue to `03_training.ipynb`. This notebook does not produce recognition scores.

The smoke-check module is written to `src/guesser_evaluation/environment_check.py` by the `%%writefile` cell, then imported and called. The committed Python module can also run from the terminal without executing this notebook:

```sh
uv run --project models/guesser/evaluation python -m guesser_evaluation.environment_check
```

From the repository root, install the workspace and launch Jupyter with the repository as its contents root. This lets notebook Markdown links reach sibling project docs. Do not append the notebooks directory as a positional argument; Jupyter then uses that directory as the server root. Open `models/guesser/evaluation/notebooks` in the file browser.

```sh
uv sync --project models/guesser/evaluation --locked
uv run --project models/guesser/evaluation jupyter lab --ServerApp.root_dir=.
```

## Development loop

Reusable Python belongs in `src/guesser_evaluation/`. The path-discovery cell imports `module_source_path(filename)`, which returns a `.py` destination under the package root. The current writer targets `environment_check.py`; for another module, set a path such as `analysis_path = str(module_source_path('analysis.py'))`, then add a separate `%%writefile {analysis_path}` cell. After writing, call `importlib.invalidate_caches()` and import it with `from guesser_evaluation.analysis import ...`; modules at the package root are importable, while a new subpackage also needs an `__init__.py`.

Treat each `%%writefile` cell as the source of truth for its generated module. Make all code changes to that module in the notebook cell, then run it to regenerate the `.py` file; direct edits to the generated file will be overwritten the next time its writer cell runs. Give every additional module its own writer cell. The test suite verifies that the environment-check writer and module stay synchronized.

The `%load_ext autoreload` and `%autoreload 2` cell is deliberately before imports. It reloads changed modules in this notebook's current kernel only; another notebook with a separate kernel and terminal Python processes are unaffected.

In [ ]:
# Reload imported modules in this kernel when cells run; other kernels are unaffected.
%load_ext autoreload
%autoreload 2

In [ ]:
# Resolve a package source path without embedding a machine-specific location.
from guesser_evaluation.notebook_tools import module_source_path

# The following writer cell regenerates only this target module.
source_path = str(module_source_path("environment_check.py"))

In [ ]:
%%writefile {source_path}
"""Smoke checks for the Guesser evaluation notebook environment."""

from __future__ import annotations

import platform
import sys
from importlib.metadata import version

import matplotlib
import numpy as np
import pandas as pd
from PIL import Image

import guesser_evaluation


def collect_environment() -> dict[str, object]:
    """Import the supported stack and exercise a small image/data operation."""
    # Fail early if Jupyter is using an unsupported Python interpreter.
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError(f"Expected Python 3.12, got {sys.version.split()[0]}")

    # Exercise grayscale image creation and conversion to a NumPy array.
    image = Image.new("L", (224, 224), color=255)
    pixels = np.asarray(image)
    if pixels.shape != (224, 224):
        raise RuntimeError(f"Unexpected image shape: {pixels.shape}")

    # Exercise tabular data creation using a small slice of the image.
    frame = pd.DataFrame({"pixel": pixels[0, :2]})
    if frame.shape != (2, 1):
        raise RuntimeError(f"Unexpected dataframe shape: {frame.shape}")

    # Report interpreter, platform, dependencies, and smoke-check shapes.
    return {
        "python": sys.version.split()[0],
        "executable": sys.executable,
        "os": platform.system(),
        "architecture": platform.machine(),
        "workspace_package": guesser_evaluation.__file__,
        "image_shape": pixels.shape,
        "dataframe_shape": frame.shape,
        "packages": {
            "ipykernel": version("ipykernel"),
            "jupyterlab": version("jupyterlab"),
            "matplotlib": matplotlib.__version__,
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "pillow": version("pillow"),
        },
    }


def main() -> None:
    """Print the environment report when run as a module from the terminal."""
    from pprint import pprint

    pprint(collect_environment())


# Keep the smoke check runnable with `python -m guesser_evaluation.environment_check`.
if __name__ == "__main__":
    main()


## Run and validate

The last code cell imports `collect_environment()` and displays the smoke-check report. The same source runs without Jupyter, and the package can be tested and linted from the repository root:

```sh
uv run --project models/guesser/evaluation python -m guesser_evaluation.environment_check
uv run --project models/guesser/evaluation python -m pytest
uv run --project models/guesser/evaluation ruff check models/guesser/evaluation/src models/guesser/evaluation/tests
```

In [ ]:
# Refresh finder caches so modules just written by %%writefile are visible.
import importlib

importlib.invalidate_caches()

# Import and run the module after executing its writer cell above.
from guesser_evaluation.environment_check import collect_environment

# Check the notebook environment only; use the training README for the model/GPU smoke test.
environment = collect_environment()
environment

## Notebook collaboration

In each clone, follow the notebook setup in the [workspace README](../README.md) to enable `nbstripout`, `nbdime`, and the scoped pre-commit hook. `nbstripout` removes cell outputs when staging; it does not replace committing the `.py` source. Commit the module and notebook as separate single-file commits.

For a notebook conflict, run `uv run --project models/guesser/evaluation nbdime mergetool`, review the cell-level result, and do not hand-edit `.ipynb` JSON. Follow the repository's conflict-approval instructions before continuing a merge or rebase.